# Comprehensive Prediction Run Analysis

This notebook provides a complete analysis of a single prediction run. It loads the `prediction_analysis.npz` file generated by `evaluate_prediction.py` and the original `test_set.npz` to visualize:

1.  **Quantitative Metrics:** Overall and per-channel R² scores.
2.  **Error Plots:** R² bar charts and error-over-time plots.
3.  **Interactive Dashboard:** A multi-panel plot for visual analysis of the Ground Truth, Prediction, and Difference fields.

### 1. Setup and Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown

from mhd_surrogate_core.plot import (
    plot_prediction_rollout_error,
    plot_r2_performance,
    plot_prediction_dashboard
)

### 2. Load Evaluation and Ground Truth Data

**Action Required:** Update the paths below to point to your specific evaluation output and the corresponding test set.

In [ ]:
# Path to the file generated by evaluate_prediction.py
eval_file_path = Path("/cephfs/users/skowronek/Documents/PhD/nuclear_fusion_cooling/prediction/mhd_surrogate_modelling/experiments/study_2_q2d_model/output/lr_1e-4_ld_4096_wr_1.0_wp_1.0_wl_100000.0_we_0.1/eval/prediction_analysis.npz")

# Path to the original test set used for the evaluation
ground_truth_path = Path("/raid/skowronek/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/interp/prep2/new_test_set.npz")

if not eval_file_path.exists() or not ground_truth_path.exists():
    print("ERROR: One or both data files not found. Please update the paths.")
else:
    with np.load(eval_file_path, allow_pickle=True) as data:
        predicted_timeseries = data['predicted_timeseries']
        difference_timeseries = data['difference_timeseries']
        channel_names = list(data['channel_names'])
        r_squared_total = data['r_squared_total']

    with np.load(ground_truth_path, allow_pickle=True) as data:
        ground_truth_timeseries = data['timeseries']
        coords = {
            'x': data['x_coords'],
            'y': data['y_coords'],
            'z': data['z_coords'],
            'labels': list(data['labels'])
        }
    
    # Configure widgets based on loaded data
    max_time_index = predicted_timeseries.shape[0] - 1
    max_x_index = predicted_timeseries.shape[1] - 1
    max_y_index = predicted_timeseries.shape[2] - 1

---

### 3. Quantitative Analysis

In [ ]:
print(f"--- PREDICTION SUMMARY ---")
print(f"Overall R² Score: {r_squared_total:.4f}\n")

print("\n--- Average Prediction Performance (R²) ---")
plot_r2_performance(eval_file_path, eval_type="Prediction")

print("\n--- Prediction Error Over Time ---")
plot_prediction_rollout_error(eval_file_path)

---

### 4. Interactive Visual Dashboard

In [ ]:
# --- Create Widgets ---
channel_dropdown = widgets.Dropdown(options=channel_names, description='Channel:', value=channel_names[0])
y_slider = widgets.IntSlider(min=0, max=max_y_index, value=max_y_index // 2, description='Y Index:')
x_slider = widgets.IntSlider(min=0, max=max_x_index, value=max_x_index // 2, description='X Index:')
time_slider = widgets.IntSlider(min=0, max=max_time_index, value=max_time_index // 2, description='Time Index:')

# --- Link Widgets to the Master Dashboard Function ---
dashboard_output = widgets.interactive_output(
    plot_prediction_dashboard,
    {
        'ground_truth_timeseries': widgets.fixed(ground_truth_timeseries),
        'predicted_timeseries': widgets.fixed(predicted_timeseries),
        'difference_timeseries': widgets.fixed(difference_timeseries),
        'coords': widgets.fixed(coords),
        'channel': channel_dropdown,
        'x_index': x_slider,
        'y_index': y_slider,
        'time_index': time_slider
    }
)

# --- Display the Dashboard ---
controls = widgets.HBox([channel_dropdown, x_slider, y_slider, time_slider])
display(widgets.VBox([controls, dashboard_output]))